
Topic: Previous Order Comparison Using LAG()

Purpose:
This script demonstrates how to compare each order with the previous order
for the same customer using the LAG() window function.

It helps for trend analysis, period-over-period comparison, transaction monitoring, and customer
behavior analysis.

In [0]:
%sql

DROP TABLE IF EXISTS orders;

CREATE TABLE orders (
    order_id INT PRIMARY KEY,
    customer_id INT NOT NULL,
    order_date DATE NOT NULL,
    total_amount DECIMAL(10, 2) NOT NULL
);

INSERT INTO orders (order_id, customer_id, order_date, total_amount) VALUES
(1001, 101, '2026-01-01', 250.00),
(1002, 101, '2026-01-05', 400.00),
(1003, 101, '2026-01-10', 150.00),
(1004, 102, '2026-01-02', 800.00),
(1005, 102, '2026-01-06', 300.00),
(1006, 102, '2026-01-12', 500.00),
(1007, 103, '2026-01-03', 700.00),
(1008, 103, '2026-01-09', 700.00);


num_affected_rows,num_inserted_rows
8,8


In [0]:
%sql

WITH order_comparison AS (
    SELECT
        order_id,
        customer_id,
        order_date,
        total_amount,
        LAG(total_amount) OVER (
            PARTITION BY customer_id
            ORDER BY order_date, order_id
        ) AS previous_order_amount
    FROM orders
)

SELECT
    order_id,
    customer_id,
    order_date,
    total_amount,
    previous_order_amount,
    total_amount - previous_order_amount AS amount_difference,
    ROUND(
        ((total_amount - previous_order_amount) / NULLIF(previous_order_amount, 0)) * 100,
        2
    ) AS percent_change
FROM order_comparison
ORDER BY
    customer_id,
    order_date;

order_id,customer_id,order_date,total_amount,previous_order_amount,amount_difference,percent_change
1001,101,2026-01-01,250.00,null,null,null
1002,101,2026-01-05,400.00,250.00,150.00,60.00
1003,101,2026-01-10,150.00,400.00,-250.00,-62.50
1004,102,2026-01-02,800.00,null,null,null
1005,102,2026-01-06,300.00,800.00,-500.00,-62.50
1006,102,2026-01-12,500.00,300.00,200.00,66.67
1007,103,2026-01-03,700.00,null,null,null
1008,103,2026-01-09,700.00,700.00,0.00,0.00


In [0]:
%sql

-- Purpose: Analyzes order trends by comparing each order's amount against the customer's immediately preceding order.


-- Define a Common Table Expression (CTE) named 'order_comparison'
-- This builds a temporary timeline of orders with historical context
WITH order_comparison AS (
    SELECT 
        order_id,
        customer_id,
        order_date,
        total_amount,
        -- LAG looks backward to grab a value from a previous row.
        -- PARTITION BY isolates calculations per customer (resets for each customer_id).
        -- ORDER BY sorts the customer's history chronologically so LAG reads correctly.
        LAG(total_amount) OVER (
            PARTITION BY customer_id 
            ORDER BY order_date, order_id
        ) AS previous_order_amount
    FROM orders
)
-- Main query to calculate variances from the prepared timeline
SELECT 
    order_id,
    customer_id,
    order_date,
    total_amount,
    previous_order_amount,
    
    -- Calculate the raw dollar difference: (Current Order) - (Previous Order)
    total_amount - previous_order_amount AS amount_difference,
    
    -- Calculate the percentage change: ((Current - Previous) / Previous) * 100
    ROUND(
        (
            (total_amount - previous_order_amount) / 
            -- NULLIF protects against division-by-zero errors by turning a 0 into a NULL
            NULLIF(previous_order_amount, 0)
        ) * 100, 
        2 -- Round the final percentage to 2 decimal places
    ) AS percent_change
FROM order_comparison
-- Sort the final output clearly by customer and order timeline
ORDER BY 
    customer_id,
    order_date;

order_id,customer_id,order_date,total_amount,previous_order_amount,amount_difference,percent_change
1001,101,2026-01-01,250.00,null,null,null
1002,101,2026-01-05,400.00,250.00,150.00,60.00
1003,101,2026-01-10,150.00,400.00,-250.00,-62.50
1004,102,2026-01-02,800.00,null,null,null
1005,102,2026-01-06,300.00,800.00,-500.00,-62.50
1006,102,2026-01-12,500.00,300.00,200.00,66.67
1007,103,2026-01-03,700.00,null,null,null
1008,103,2026-01-09,700.00,700.00,0.00,0.00
